# Phase 1: QLoRA fine-tuning of Qwen3.5-4B on the MMCultureQA train split (SemEval 2027 Task 6, Task 2)

Trains two adapters at once (S1: lr 1e-4 on GPU 0, S2: lr 2e-4 on GPU 1), then answers the first 500 dev items per track
with each adapter, using the exact prompt, pixel budget and token cap of the zero-shot control in Phase 2.
Everything is generated here; scoring is done locally. Download `phase1_results.zip` from the Output tab.

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, Internet **on**. Do a smoke run first (`LIMIT = 4`, about 40 min),
then set `LIMIT = None` and **Save Version -> Save & Run All (Commit)**. A commit keeps running with the browser closed.
Run Phase 2 in a separate session afterwards (one account: they cannot run at the same time).

## 1. Config

In [ ]:
REPO_URL = "https://github.com/hassan-31x/mmcqa-experiments.git"
BRANCH = "main"
LIMIT = 4              # smoke test: 4 dev items per track and a 10-step training. Set to None for the full run.
BUDGET_HOURS = 11.0    # the run stops starting new work after this (Kaggle's limit is 12 h)
TRAIN_HOURS = 6.0      # training time per adapter; the rest of the budget answers dev500
DEVTEST = False        # True: also answer devtest with the better adapter if >= 3 h remain
INSTALL_FLA = True     # install flash-linear-attention (Triton kernels); the smoke run shows if it is faster on T4
WORK = "/kaggle/working/results"

## 2. Code and dependencies

In [ ]:
import glob, os, subprocess, sys
CODE = "/kaggle/working/mmcqa-experiments"
if not os.path.isdir(CODE):
    local = glob.glob("/kaggle/input/**/mmcqa_exp/__init__.py", recursive=True)    # a Kaggle Dataset copy also works
    if local:
        subprocess.run(["cp", "-r", os.path.dirname(os.path.dirname(local[0])), CODE], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
!nvidia-smi -L
!pip install -q -r {CODE}/requirements/qwen.txt
if globals().get("INSTALL_FLA", False):        # optional fast kernels for Qwen3.5's linear-attention layers
    !pip install -q flash-linear-attention
    print(subprocess.run([sys.executable, "-c", "import fla; print('fla', fla.__version__)"], capture_output=True, text=True))
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|peft|bitsandbytes|accelerate|huggingface-hub|tokenizers) "

In [ ]:
try:                                   # optional: a Kaggle secret HF_TOKEN avoids anonymous download limits
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: continuing with anonymous downloads")

## 3. Train both adapters, then answer dev500
Progress is printed by the orchestrator. Logs: `results/logs/`.

In [ ]:
args = f"--workdir {WORK} --budget-hours {BUDGET_HOURS} --train-hours {TRAIN_HOURS}"
if LIMIT: args += f" --limit {LIMIT}"
if DEVTEST: args += " --devtest"
!cd {CODE} && python -m mmcqa_exp.phase1 {args}

## 4. What happened

In [ ]:
import json, glob
for p in sorted(glob.glob(f"{WORK}/logs/sft_*_sizing.json")) + sorted(glob.glob(f"{WORK}/logs/sft_*_summary.json")):
    print(os.path.basename(p), json.dumps(json.load(open(p)), indent=1)[:900])
for p in sorted(glob.glob(f"{WORK}/logs/sft_*_failed.json")):
    print("FAILED:", p, json.load(open(p))["error"])
print(open(f"{WORK}/logs/phase1_completion.json").read()[:3000])
for p in sorted(glob.glob(f"{WORK}/predictions/sft_*/dev_arz.jsonl")):
    print(p); [print("  ", json.loads(l)["question"], "->", json.loads(l)["answer"]) for l in open(p).readlines()[:3]]

## 5. Package
Adapters, logs, predictions and submission-format files in one zip. Images and the starter-kit clone are excluded.

In [ ]:
ZIPNAME = "phase1_results.zip"

import shutil, pathlib
media = pathlib.Path(os.environ.get("MMCQA_MEDIA", "/kaggle/temp/mmcqa_media"))
for tmp in (WORK + "/mmcultureqa", "/kaggle/working/mmcqa_media", "/kaggle/working/mmcqa_envs", str(media), str(media.parent / "mmcqa_envs")):
    shutil.rmtree(tmp, ignore_errors=True)
!cd /kaggle/working && zip -qr {ZIPNAME} results && ls -lh {ZIPNAME}